# Imports and Database Connection

In [24]:
import re
from mongoengine import connect, disconnect
from pycoshark.mongomodels import Project, VCSSystem, Commit, File, FileAction, Hunk, Refactoring, IssueSystem, Issue, IssueComment, MailingList, Message, Event
from pycoshark.utils import create_mongodb_uri_string
import pandas as pd
from collections import defaultdict

# You may have to update this dict to match your DB credentials
credentials = {'db_user': '',
               'db_password': '',
               'db_hostname': 'localhost',
               'db_port': 27017,
               'db_authentication_database': '',
               'db_ssl_enabled': False}

uri = create_mongodb_uri_string(**credentials)

disconnect(alias='default')

connect('smartshark_2_1', host=uri, alias='default')

MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True, read_preference=Primary())

In [31]:
# Print all distinct values for the 'status' field in the Issue collection
distinct_statuses = Issue.objects.distinct('status')
print("\nDistinct statuses in Issue collection:")
for status in distinct_statuses:
    print(status)

distinct_events = Event.objects.distinct('status')
print("\nDistinct statuses in Event collection:")
for event in distinct_events:
    print(event)

# time_spent_events = Event.objects(status='timespent')
# print("\nEvents with status 'time_spent':")
# for event in time_spent_events:
#     print(event)


Distinct statuses in Issue collection:
Blocked
Closed
In Progress
Open
Patch Available
Reopened
Resolved

Distinct statuses in Event collection:
Admin-Comment-Bulk Update Author
Artifact ID
Assignee
Attachment
Attachment License
Browser Version
Bug behavior facts
Bugzilla Id
Bugzilla Import Key
Comment
Complexity
Customer Request Type
Customer(s)
Database
Derby Categories
Derby Info
Description
Docs Text
Epic Child
Epic Link
Epic Name
Epic Status
Estimated Complexity
External issue ID
External issue URL
Fix For
Fix priority
Flagged
Flags
Group ID
Hadoop Flags
INFRA-Members
Issue & fix info
Issue Type
Issue-Comment-Bulk Replace String 
Issue-Comment-Bulk Update Author
JDK Version
Key
Level of effort
O/S Version
Other Info
Parent Issue
Patch
Patch Info
Patch Submitted
Patch attached
Priority
Project
ProjectImport
Rank
Regression
Release Note
RemoteIssueLink
Reproduced In
Request participants
Resolution
Reviewer
Severity
Sprint
Status
Summary
Tags
Target Version/s
Testcase included
Urgen

# Choosing projects

In [3]:
def get_project_statistics_dataframe():
    project_results = {
        p.id: {
            "Project ID": str(p.id),
            "Project Name": p.name,
            "Total Issues": 0, "Open Issues": 0, "Closed Issues": 0, "Resolved Issues": 0,
            "Total Files": 0, "Total File Actions": 0,
            "Linked Issues": 0, "Linked Percentage": 0.0
        }
        for p in Project.objects.only('id', 'name')
    }

    issue_sys_to_proj = {
        sys.id: sys.project_id for sys in IssueSystem.objects.only('id', 'project_id')
    }
    vcs_to_proj = {
        vcs.id: vcs.project_id for vcs in VCSSystem.objects.only('id', 'project_id')
    }
    file_to_vcs = {
        f.id: f.vcs_system_id for f in File.objects.only('id', 'vcs_system_id')
    }
    issue_to_proj = {
        issue.id: issue_sys_to_proj[issue.issue_system_id]
        for issue in Issue.objects.only('id', 'issue_system_id')
        if issue.issue_system_id in issue_sys_to_proj
    }

    # Process all issues
    issue_pipeline = [
        {"$group": {
            "_id": {"system": "$issue_system_id", "status": "$status"},
            "count": {"$sum": 1}
        }}
    ]
    for result in Issue.objects.aggregate(issue_pipeline):
        sys_id = result['_id'].get('system')
        status = result['_id'].get('status', '').lower() if result['_id'].get('status') else ''
        count = result['count']

        if sys_id in issue_sys_to_proj:
            pid = issue_sys_to_proj[sys_id]
            project_results[pid]["Total Issues"] += count
            
            if status == "open":
                project_results[pid]["Open Issues"] += count
            elif status == "closed":
                project_results[pid]["Closed Issues"] += count
            elif status == "resolved":
                project_results[pid]["Resolved Issues"] += count

    # Process all files
    file_pipeline = [{"$group": {"_id": "$vcs_system_id", "count": {"$sum": 1}}}]
    for result in File.objects.aggregate(file_pipeline):
        if result['_id'] in vcs_to_proj:
            project_results[vcs_to_proj[result['_id']]]["Total Files"] += result['count']

    # Process all file actions
    action_pipeline = [{"$group": {"_id": "$file_id", "count": {"$sum": 1}}}]
    for result in FileAction.objects.aggregate(action_pipeline):
        if result['_id'] in file_to_vcs:
            vcs_id = file_to_vcs[result['_id']]
            if vcs_id in vcs_to_proj:
                project_results[vcs_to_proj[vcs_id]]["Total File Actions"] += result['count']

    # Process all issues linked to commits
    unique_linked_issues = set()
    for commit in Commit.objects(linked_issue_ids__not__size=0).only('linked_issue_ids'):
        for issue_id in commit.linked_issue_ids:
            unique_linked_issues.add(issue_id)

    for issue_id in unique_linked_issues:
        if issue_id in issue_to_proj:
            pid = issue_to_proj[issue_id]
            project_results[pid]["Linked Issues"] += 1

    # Calculate linked percentage
    for pid, stats in project_results.items():
        if stats["Total Issues"] > 0:
            stats["Linked Percentage"] = round((stats["Linked Issues"] / stats["Total Issues"]) * 100, 2)
              
    # Create DataFrame table to display results
    data_list = list(project_results.values())
    df = pd.DataFrame(data_list)
    df = df.sort_values(
        by=['Total Issues', 'Linked Percentage', 'Total File Actions', 'Total Files'], 
        ascending=[False, False, False, False]
    )
    
    return df.reset_index(drop=True)

df_projects = get_project_statistics_dataframe()
df_projects

,Project ID,Project Name,Total Issues,Open Issues,Closed Issues,Resolved Issues,Total Files,Total File Actions,Linked Issues,Linked Percentage
0,5e84988880c620a732667310,maven,13238,656,10537,14,9215,59841,3864,29.19
1,5e71c4d43c63727691322420,activemq,7324,716,1681,4798,24709,154962,3622,49.45
2,5bdc03c30da3ef21a62b756c,derby,7027,1213,5716,49,11183,75682,3837,54.60
3,5b0fa95e0da3ef431af80833,kafka,7007,1499,561,4651,5412,52430,3215,45.88
4,5bd022e30da3ef1a10738359,nifi,5813,1586,98,3913,17074,154910,3220,55.39
...,...,...,...,...,...,...,...,...,...,...
72,5bee7a2a0da3ef21a58b337e,commons-imaging,220,50,13,152,1766,10473,69,31.36
73,5bee7a9d0da3ef21a3a094b1,commons-jcs,194,13,163,17,3960,16477,104,53.61
74,5bee79dc0da3ef21a58b337d,commons-digester,194,23,150,16,1923,9730,54,27.84
75,60a24a978d56179891397c71,freemarker,184,49,102,31,7667,57455,44,23.91


Based on these results the following projects have been chosen to perform the experiment on:
- ActiveMQ
- Nifi
- Manifoldcf
- Storm
- Jena

# Experiment

In [4]:
# Map the projects to their system_ids
target_project_names = ['activemq', 'nifi', 'manifoldcf', 'storm', 'jena']

vcs_to_proj_name = defaultdict()
proj_to_vcs = defaultdict()

for p in Project.objects(name__in=target_project_names).only('id', 'name'):
    vcs = VCSSystem.objects(project_id=p.id).only('id').first()
    if vcs:
        vcs_to_proj_name[vcs.id] = p.name
        proj_to_vcs[p.name] = vcs.id

target_vcs_ids = list(vcs_to_proj_name.keys())

### Retrieve hotspots
Retrieving the files that are adjusted the most of each project. The issues that are connected to these files are labeled as hotspot issues.

In [20]:
vcs_to_commits = defaultdict()
commit_to_issues = defaultdict()
file_to_commit = defaultdict()

project_file_freq = defaultdict() # How many times a file was changed for a project
project_hotspot_issues = defaultdict(set) # List of issues linked to commits that changed a hotspot file for a project
project_normal_issues = defaultdict(set) # List of issues linked to commits that changed a non-hotspot file for a project

NUMBER_OF_HOTSPOTS = 50

for p in Project.objects(name__in=target_project_names).only('id', 'name'):
    vcs_id = proj_to_vcs.get(p.name)
    if vcs_id:
        project_file_freq[vcs_id] = {}
        # Link all commits related to the vcs
        commits = Commit.objects(vcs_system_id=vcs_id).only('id', 'linked_issue_ids')
        vcs_to_commits[vcs_id] = [c.id for c in commits]
        for c in commits:
            # Link each commit to its linked issues and each issue to its linked commits
            commit_to_issues[c.id] = c.linked_issue_ids

            # Link each commit to its file actions
            file_actions = FileAction.objects(commit_id=c.id).only('id', 'file_id', 'commit_id')

            # Count the file actions for each file for the vcs
            for fa in file_actions:
                file_to_commit.setdefault(fa.file_id, []).append(fa.commit_id)
                project_file_freq[vcs_id][fa.file_id] = project_file_freq[vcs_id].get(fa.file_id, 0) + 1

# Calculate and print the top NUMBER_OF_HOTSPOTS hotspot files for each project
for vcs_id, freq in project_file_freq.items():
    project_name = vcs_to_proj_name.get(vcs_id, "Unknown Project")
    print(f"\nTop hotspots for project '{project_name}':")

    accumulated_hotspot_issues = set()
    accumulated_normal_issues = set()
    # Sort the files on freq
    project_file_freq[vcs_id] = sorted(freq.items(), key=lambda x: x[1], reverse=True)

    # Sort the hotspots and add related issues from each hotspot file
    for file_id, count in project_file_freq[vcs_id][:NUMBER_OF_HOTSPOTS]:
        file_issues = [
            issue_id 
            for commit_id in file_to_commit.get(file_id, []) 
            for issue_id in commit_to_issues.get(commit_id, [])
        ]
        accumulated_hotspot_issues.update(file_issues)

    # Sort the non-hotspots and add related issues from each non-hotspot file
    for file_id, count in project_file_freq[vcs_id][NUMBER_OF_HOTSPOTS:]:
        file_issues = [
            issue_id 
            for commit_id in file_to_commit.get(file_id, []) 
            for issue_id in commit_to_issues.get(commit_id, [])
            if issue_id not in accumulated_hotspot_issues  # Don't count issues linked to hotspot files twice
        ]
        accumulated_normal_issues.update(file_issues)

    project_hotspot_issues[vcs_id] = accumulated_hotspot_issues
    project_normal_issues[vcs_id] = accumulated_normal_issues
    print(f"Total unique linked issues across top {NUMBER_OF_HOTSPOTS} hotspot files for VCS {vcs_id}: {len(project_hotspot_issues[vcs_id])}")
    print(f"Total unique linked issues across top {NUMBER_OF_HOTSPOTS} non-hotspot files for VCS {vcs_id}: {len(project_normal_issues[vcs_id])}")


Top hotspots for project 'activemq':
Total unique linked issues across top 50 hotspot files for VCS 5e71ce99a18d9712488b3a92: 1203
Total unique linked issues across top 50 non-hotspot files for VCS 5e71ce99a18d9712488b3a92: 2417

Top hotspots for project 'storm':
Total unique linked issues across top 50 hotspot files for VCS 5b7ffe9530a71b06bc70c038: 1607
Total unique linked issues across top 50 non-hotspot files for VCS 5b7ffe9530a71b06bc70c038: 149

Top hotspots for project 'nifi':
Total unique linked issues across top 50 hotspot files for VCS 5bd0268235e3ea2b7bbfdbae: 858
Total unique linked issues across top 50 non-hotspot files for VCS 5bd0268235e3ea2b7bbfdbae: 2360

Top hotspots for project 'jena':
Total unique linked issues across top 50 hotspot files for VCS 5cb58b44504acf99a4cf1827: 255
Total unique linked issues across top 50 non-hotspot files for VCS 5cb58b44504acf99a4cf1827: 829

Top hotspots for project 'manifoldcf':
Total unique linked issues across top 50 hotspot files 

Check properties of events for the selected issues for reliability

In [42]:
for vcs_id, project_name in vcs_to_proj_name.items():
    print(f"\nFirst 10 hotspot-linked issues for project '{project_name}':")
    issues_list = list(project_hotspot_issues.get(vcs_id, set()))
    for issue in issues_list[:10]:
        print(issue)
        timespent = Event.objects(issue_id=issue, status='timespent')
        for ts in timespent:
            print(ts)
        resolution = Event.objects(issue_id=issue, status='resolution')
        for res in resolution:
            print(res)
        status = Event.objects(issue_id=issue, status='status')
        for st in status:
            print(st)
    break


First 10 hotspot-linked issues for project 'activemq':
5f0d753e5e36a95000e3167c
external_id: 18583215%%1, issue_id: 5f0d753e5e36a95000e3167c, created_at: 2020-02-06 20:43:21.129000, status: timespent, author_id: 5b2a02f230a71b06bcffebe0, old_value: 600, new_value: 1200, commit_id: None
external_id: 18556215%%2, issue_id: 5f0d753e5e36a95000e3167c, created_at: 2020-01-28 13:33:57.640000, status: timespent, author_id: 5b2a02f230a71b06bcffebe0, old_value: None, new_value: 600, commit_id: None
external_id: 18583221%%0, issue_id: 5f0d753e5e36a95000e3167c, created_at: 2020-02-06 20:44:33.470000, status: resolution, author_id: 5b3b407f30a71b06bc85924a, old_value: None, new_value: Fixed, commit_id: None
external_id: 18583221%%1, issue_id: 5f0d753e5e36a95000e3167c, created_at: 2020-02-06 20:44:33.470000, status: status, author_id: 5b3b407f30a71b06bc85924a, old_value: Open, new_value: Resolved, commit_id: None
5f0d75565e36a95000e318b6
external_id: 18641123%%1, issue_id: 5f0d75565e36a95000e318b6,

Calculate the cycle times of the issues that contain hotspot files by looking at the difference between created_at and updated_at for closed or resolved issues assuming that the latest updated_at correlates to the closing time of an issue

In [ ]:
import pandas as pd

cycle_time_results = []

for p in Project.objects(name__in=target_project_names).only('id', 'name'):
    vcs_id = proj_to_vcs.get(p.name)
    if not vcs_id:
        continue
        
    # Convert hotspot list to a set for O(1) lookup speeds
    hotspot_set = set(project_hotspot_issues.get(vcs_id, []))
    normal_set = set(project_normal_issues.get(vcs_id, []))

    issue_systems = [sys.id for sys in IssueSystem.objects(project_id=p.id).only('id')]
    
    # Restrict the query to completed issues to avoid calculating cycle times for open tasks
    issues = Issue.objects(
        issue_system_id__in=issue_systems,
        status__in=['Closed', 'Resolved']
    ).only('id', 'created_at', 'updated_at')
    
    # Strip the heavy ODM layer immediately and build a flat dictionary
    data = []
    for issue in issues:
        if (issue.id in hotspot_set or issue.id in normal_set) and issue.created_at and issue.updated_at:
            data.append({
                'issue_id': issue.id,
                'created_at': issue.created_at,
                'updated_at': issue.updated_at,
                'is_hotspot': issue.id in hotspot_set
            })
            
    if not data:
        print(f"No valid closed/resolved issues found for {p.name}")
        continue
        
    df = pd.DataFrame(data)
    
    # Execute cycle time calculation via vectorized Pandas operations
    df['cycle_time_hours'] = (df['updated_at'] - df['created_at']).dt.total_seconds() / 3600.0
    
    # Filter out data entry anomalies (e.g., negative time)
    df = df[df['cycle_time_hours'] > 0] 
    
    # Create masks for the two cohorts
    is_hotspot = df['is_hotspot'] == True
    
    # Use median instead of mean to prevent long-tail outliers from destroying the metric
    hotspot_median = df[is_hotspot]['cycle_time_hours'].median()
    other_median = df[~is_hotspot]['cycle_time_hours'].median()
    hotspot_mean = df[is_hotspot]['cycle_time_hours'].mean()
    other_mean = df[~is_hotspot]['cycle_time_hours'].mean()
    
    cycle_time_results.append({
        'Project': p.name,
        'Hotspot Median Cycle (Hours)': hotspot_median,
        'Other Median Cycle (Hours)': other_median,
        'Hotspot Mean Cycle (Hours)': hotspot_mean,
        'Other Mean Cycle (Hours)': other_mean,
        'Hotspot Issue Count': is_hotspot.sum(),
        'Other Issue Count': (~is_hotspot).sum()
    })

df_cycle_times = pd.DataFrame(cycle_time_results)
display(df_cycle_times)

,Project,Hotspot Median Cycle (Hours),Other Median Cycle (Hours),Hotspot Mean Cycle (Hours),Other Mean Cycle (Hours),Hotspot Issue Count,Other Issue Count
0,activemq,617.900505,348.848090,5479.880025,4660.719272,1185,2395
1,storm,1127.521966,143.013666,3685.111144,1310.489377,1584,143
2,nifi,721.534589,249.332962,2192.313637,1352.415227,849,2343
3,jena,1426.775399,1369.576189,6138.809142,4812.504308,248,825
4,manifoldcf,24.925378,299.379071,963.477412,2653.957724,1100,68


### Retrieve large files 

### Retrieve clumps